# Stage 2: Supervised Fine-Tuning (SFT)
This notebook executes the Supervised Fine-Tuning phase, loading either the adapter checkpoint from Stage 1 or starting directly from the base model, mapping SFT templates to teach the model how to follow instruction-based user support prompts.

To load a previously fine-tuned model (either a full model or an adapter), you'll typically use the `transformers` library, potentially with `peft` if you used LoRA or similar adapter-based tuning.

First, make sure you have the necessary libraries installed:

In [1]:
# Install libraries in Google Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft transformers accelerate bitsandbytes
!pip install unsloth_zoo
!pip install datasets
!pip install trl

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-v67u5sdi/unsloth_2f85bd5d942c4bd3a451ef56b55caa05
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-v67u5sdi/unsloth_2f85bd5d942c4bd3a451ef56b55caa05
  Resolved https://github.com/unslothai/unsloth.git to commit 9fa6fd40e1a227a6c77b7e64d33dcfe3d5f617cd
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 133.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 24.6 MB/s eta 0:00:00
   

In [2]:
from google.colab import userdata
from huggingface_hub import login, upload_folder, create_repo
import os

# 2. Push that local merged folder directly to Hugging Face
# Ensure your Hugging Face token is stored as a Colab secret named 'HF_TOKEN'
hf_token = userdata.get('HF_TOKEN')

# Now you can use hf_token in your login function or other operations
login(token=hf_token)
print("Logged in to Hugging Face successfully!")

Logged in to Hugging Face successfully!


In [2]:
import torch
from unsloth import FastLanguageModel
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load the base model first
model, tokenizer = FastLanguageModel.from_pretrained(
     model_name = "Bhargav1/qwen2.5-7b-stage1-merged",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


# Prepare the model for further PEFT operations (e.g., if resuming training)
# This step is crucial to make the LoRA layers trainable for SFT.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [3]:
prompt_format = """Below is an instruction that describes a customer support scenario for practiceyourspeech.com. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{response}"""

EOS_TOKEN = tokenizer.eos_token
def format_prompts(examples):
    instructions = examples["instruction"]
    responses    = examples["response"]
    texts = []
    for inst, resp in zip(instructions, responses):
        text = prompt_format.format(instruction=inst, response=resp) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

In [4]:
from datasets import load_dataset

# Load the instruction dataset (upload 'instruction_dataset.jsonl' via the sidebar)
dataset = load_dataset("json", data_files="/content/drive/MyDrive/AIML-2026/instruction_dataset.jsonl", split="train")
dataset = dataset.map(format_prompts, batched = True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/105 [00:00<?, ? examples/s]

In [5]:

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1, # Changed from 2 to 1 to avoid PicklingError with multiprocessing
    packing = False,
     args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no", # Disable saving to bypass pickling issues during checkpoint creation
        report_to = "none", # Disable reporting to avoid potential issues with logger pickling
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/105 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 105 | Num Epochs = 9 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,3.839963
10,2.508781
15,1.342829
20,1.155156
25,1.082849
30,0.917999
35,0.825204
40,0.672394
45,0.584515
50,0.483475


In [6]:
# Inference test
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    prompt_format.format(
        instruction = "My payment went through, but I am still not upgraded. Why?",
        response = "",
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 150, use_cache = True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Below is an instruction that describes a customer support scenario for practiceyourspeech.com. Write a response that appropriately completes the request.

### Instruction:
My payment went through, but I am still not upgraded. Why?

### Response:
First, check that your role is set to 'student' or 'instructor' in the SpeechUsers table. Verify that Stripe processed the invoice and updated the subscription date.


In [8]:
from google.colab import userdata
from huggingface_hub import login, upload_folder, create_repo
import os

# 2. Push that local merged folder directly to Hugging Face
# Ensure your Hugging Face token is stored as a Colab secret named 'HF_TOKEN'
hf_token = userdata.get('HF_TOKEN')

# Now you can use hf_token in your login function or other operations
login(token=hf_token)
print("Logged in to Hugging Face successfully!")

model.push_to_hub_merged(
    "Bhargav1/qwen2.5-7b-stage2-merged",
    tokenizer,
    save_method="merged_16bit",
    token=hf_token
)

Logged in to Hugging Face successfully!


Unsloth: Restored added_tokens_decoder metadata in Bhargav1/qwen2.5-7b-stage2-merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   3%|2         | 31.9MB / 1.09GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [10:18<00:00, 154.51s/it]


Unsloth: Merge process complete. Saved to `/content/Bhargav1/qwen2.5-7b-stage2-merged`
